# 🛒 E-Commerce: Identificar os Perfis dos Consumidores

**Cliente:** Everything Plus, loja online de utensílios domésticos.
**Pergunta de negócio:** quem são os nossos consumidores e como o histórico de compras pode orientar campanhas de marketing mais personalizadas?

**Etapas deste notebook**
1. Carga e diagnóstico de qualidade dos dados
2. Limpeza com rastreabilidade (quantas linhas saem em cada regra)
3. Análise exploratória (KPIs, sazonalidade)
4. Segmentação por **produto** e por **frequência de compra**
5. Teste de hipótese: *clientes recorrentes gastam mais do que clientes de compra única?*
6. Resumo automático dos resultados e limitações

## Dicionário de dados

Arquivo `data/ecommerce_dataset_us.csv` (separador de tabulação).

| Coluna | Descrição |
|---|---|
| `InvoiceNo` | Identificador do pedido (um pedido tem várias linhas, uma por item). Começa com **C** em cancelamentos |
| `StockCode` | Identificador do item |
| `Description` | Nome do item |
| `Quantity` | Quantidade do item |
| `InvoiceDate` | Data e hora do pedido |
| `UnitPrice` | Preço por item |
| `CustomerID` | Identificador do cliente (pode estar ausente) |

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

try:
    from IPython.display import display
except ImportError:  # fora do Jupyter
    display = print

pd.set_option('display.float_format', '{:,.2f}'.format)
plt.rcParams.update({'figure.figsize': (10, 5), 'axes.grid': True, 'grid.alpha': 0.3})

DATA_PATH = os.path.join('data', 'ecommerce_dataset_us.csv')
os.makedirs('images', exist_ok=True)   # gráficos salvos aqui (usados no README)

## 1. Carga e diagnóstico dos dados

In [ ]:
df_raw = pd.read_csv(DATA_PATH, sep='\t')
print(f'{df_raw.shape[0]:,} linhas x {df_raw.shape[1]} colunas')
df_raw.info()
df_raw.head()

In [ ]:
inv = df_raw['InvoiceNo'].astype(str)
sc = df_raw['StockCode'].astype(str)
nao_produto = ~sc.str.match(r'^\d{5}')   # produtos têm código iniciado por 5 dígitos

nulos = pd.DataFrame({'nulos': df_raw.isna().sum(), '% nulos': df_raw.isna().mean() * 100})
display(nulos)

problemas = pd.Series({
    'Linhas totais': len(df_raw),
    'Cancelamentos (InvoiceNo começa com "C")': inv.str.startswith('C').sum(),
    'Quantity <= 0': (df_raw['Quantity'] <= 0).sum(),
    'UnitPrice <= 0': (df_raw['UnitPrice'] <= 0).sum(),
    'Linhas duplicadas': df_raw.duplicated().sum(),
    'StockCode fora do padrão de produto': nao_produto.sum(),
})
display(problemas.to_frame('quantidade'))

print('Códigos que não são produto (frete, taxas, ajustes etc.) mais frequentes:')
display(df_raw.loc[nao_produto, ['StockCode', 'Description']].value_counts().head(15).to_frame('linhas'))

### Achados do diagnóstico e decisões de limpeza

- **Sem `CustomerID`:** não dá para analisar comportamento por cliente, então essas linhas saem da segmentação.
- **Cancelamentos (`InvoiceNo` com "C") e quantidades/preços ≤ 0:** são devoluções ou ajustes, não vendas.
- **Compras posteriormente canceladas:** o cancelamento tem sinal negativo, mas a **compra original continua na base** e inflaria a receita. Removo a compra original quando existe um cancelamento do mesmo cliente, mesmo item e mesma quantidade (visão de receita líquida). É uma decisão de negócio, documentada aqui.
- **Frete, taxas e ajustes** (`POSTAGE`, `DOTCOM POSTAGE`, `BANK CHARGES` etc.) não são produtos e distorcem o ranking de itens.
- **Duplicatas exatas** saem.

## 2. Limpeza com rastreabilidade

In [ ]:
df = df_raw.copy()
df['InvoiceNo'] = df['InvoiceNo'].astype(str)
df['StockCode'] = df['StockCode'].astype(str)

try:
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%m/%d/%Y %H:%M')
except ValueError:
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# chaves das compras que foram canceladas depois (mesmo cliente, item e quantidade)
canc = df[df['InvoiceNo'].str.startswith('C')].dropna(subset=['CustomerID'])
cancel_keys = (canc.assign(Quantity=-canc['Quantity'])
                   [['CustomerID', 'StockCode', 'Quantity']].drop_duplicates())

def remover_compras_canceladas(d):
    m = d.merge(cancel_keys, on=['CustomerID', 'StockCode', 'Quantity'], how='left', indicator=True)
    return m.loc[m['_merge'] == 'left_only'].drop(columns='_merge')

regras = [
    ('Sem CustomerID',                      lambda d: d.dropna(subset=['CustomerID'])),
    ('Cancelamentos (InvoiceNo com "C")',   lambda d: d[~d['InvoiceNo'].str.startswith('C')]),
    ('Quantity <= 0 ou UnitPrice <= 0',     lambda d: d[(d['Quantity'] > 0) & (d['UnitPrice'] > 0)]),
    ('Frete, taxas e ajustes (não produto)', lambda d: d[d['StockCode'].str.match(r'^\d{5}')]),
    ('Compras depois canceladas',           remover_compras_canceladas),
    ('Linhas duplicadas',                   lambda d: d.drop_duplicates()),
]

funil = [{'etapa': 'Base original', 'removidas': 0, 'restantes': len(df)}]
for nome, regra in regras:
    antes = len(df)
    df = regra(df)
    funil.append({'etapa': nome, 'removidas': antes - len(df), 'restantes': len(df)})

df['CustomerID'] = df['CustomerID'].astype(int)
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df = df.reset_index(drop=True)

display(pd.DataFrame(funil))
print(f'Base final: {len(df):,} linhas ({len(df) / len(df_raw):.1%} da original)')

## 3. Análise exploratória

In [ ]:
pedidos = (df.groupby('InvoiceNo')
             .agg(data=('InvoiceDate', 'min'), cliente=('CustomerID', 'first'),
                  valor=('TotalPrice', 'sum'), itens=('Quantity', 'sum'))
             .reset_index())

kpis = {
    'Período': f"{df['InvoiceDate'].min():%d/%m/%Y} a {df['InvoiceDate'].max():%d/%m/%Y}",
    'Clientes únicos': f"{df['CustomerID'].nunique():,}",
    'Pedidos únicos': f"{len(pedidos):,}",
    'Receita total (US$)': f"{df['TotalPrice'].sum():,.2f}",
    'Ticket médio por pedido (US$)': f"{pedidos['valor'].mean():,.2f}",
    'Ticket mediano por pedido (US$)': f"{pedidos['valor'].median():,.2f}",
    'Itens por pedido (mediana)': f"{pedidos['itens'].median():,.0f}",
}
for k, v in kpis.items():
    print(f'{k:<34} {v}')

display(df[['Quantity', 'UnitPrice', 'TotalPrice']].describe())

### Sazonalidade: receita mensal e diária

Os meses das pontas podem estar incompletos (a base não começa no dia 1 nem termina no último dia do mês).

In [ ]:
serie = pedidos.set_index('data')

mensal = serie['valor'].resample('MS').sum()
fig, ax = plt.subplots()
ax.bar(mensal.index.strftime('%Y-%m'), mensal.values, color='#1f77b4')
ax.set_title('Receita mensal (US$)')
ax.set_ylabel('Receita (US$)')
plt.xticks(rotation=45)
plt.savefig('images/receita_mensal.png', dpi=150, bbox_inches='tight')
plt.show()

diario = serie['valor'].resample('D').agg(['size', 'sum']).rename(columns={'size': 'pedidos', 'sum': 'receita'})
fig, ax = plt.subplots()
ax.plot(diario.index, diario['receita'], alpha=0.35, label='Receita diária')
ax.plot(diario.index, diario['receita'].rolling(7, center=True).mean(), linewidth=2, label='Média móvel de 7 dias')
ax.set_title('Receita diária (US$)')
ax.set_ylabel('Receita (US$)')
ax.legend()
plt.savefig('images/receita_diaria.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Segmentação por produto

Ranking por **receita** dos itens que são produtos (frete e taxas já foram removidos).

In [ ]:
produtos = (df.groupby('StockCode')
              .agg(descricao=('Description', 'first'), receita=('TotalPrice', 'sum'),
                   unidades=('Quantity', 'sum'), pedidos=('InvoiceNo', 'nunique'))
              .sort_values('receita', ascending=False))

top10 = produtos.head(10)
share_top10 = top10['receita'].sum() / produtos['receita'].sum()
print(f'Os 10 maiores itens concentram {share_top10:.1%} da receita (de {len(produtos):,} itens).')
display(top10)

fig, ax = plt.subplots()
ax.barh(top10['descricao'][::-1], top10['receita'][::-1], color='#2ca02c')
ax.set_title('Top 10 produtos por receita (US$)')
plt.savefig('images/top10_produtos.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Segmentação por frequência de compra

Frequência = número de **pedidos distintos** (`InvoiceNo` únicos) por cliente. Os cortes dos grupos são uma escolha de negócio e podem ser ajustados.

In [ ]:
clientes = (df.groupby('CustomerID')
              .agg(n_pedidos=('InvoiceNo', 'nunique'), gasto_total=('TotalPrice', 'sum'),
                   ultima_compra=('InvoiceDate', 'max')))
clientes['ticket_medio'] = clientes['gasto_total'] / clientes['n_pedidos']
clientes['segmento'] = pd.cut(clientes['n_pedidos'], bins=[0, 1, 3, 9, np.inf],
                              labels=['1 pedido', '2-3 pedidos', '4-9 pedidos', '10+ pedidos'])

seg = (clientes.groupby('segmento', observed=True)
               .agg(clientes=('gasto_total', 'size'), receita=('gasto_total', 'sum'),
                    ticket_medio=('ticket_medio', 'mean')))
seg['% clientes'] = seg['clientes'] / seg['clientes'].sum() * 100
seg['% receita'] = seg['receita'] / seg['receita'].sum() * 100
display(seg)

seg[['% clientes', '% receita']].plot.bar(rot=0)
plt.title('Participação de cada segmento: clientes x receita (%)')
plt.ylabel('%')
plt.savefig('images/segmentos_clientes.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Teste de hipótese

**Pergunta:** clientes recorrentes (2 ou mais pedidos) gastam mais do que clientes de compra única (1 pedido)?

- **H₀:** a média do grupo recorrente é **menor ou igual** à do grupo de compra única.
- **H₁:** a média do grupo recorrente é **maior**.
- Nível de significância: **α = 0,05**.

**Duas métricas, de propósito:**
1. `gasto_total` por cliente. Atenção: quem compra mais vezes tende a gastar mais *por construção*, então essa métrica favorece a hipótese.
2. `ticket_medio` (gasto por pedido). É a comparação mais justa: o cliente recorrente gasta mais **em cada compra**?

**Testes:** Welch (teste t que não assume variâncias iguais) e Mann-Whitney (não assume normalidade). O gasto por cliente é muito assimétrico, então o segundo é a verificação de robustez. O tamanho de efeito (d de Cohen) mostra se a diferença importa na prática, não só se é estatisticamente significativa.

In [ ]:
recorrente = clientes[clientes['n_pedidos'] > 1]
unico = clientes[clientes['n_pedidos'] == 1]
ALFA = 0.05

def comparar(metrica):
    a, b = recorrente[metrica], unico[metrica]
    t, p_welch = stats.ttest_ind(a, b, equal_var=False, alternative='greater')
    _, p_mw = stats.mannwhitneyu(a, b, alternative='greater')
    desvio_comb = np.sqrt(((len(a) - 1) * a.var() + (len(b) - 1) * b.var()) / (len(a) + len(b) - 2))
    return {'n_recorrentes': len(a), 'n_unicos': len(b),
            'media_rec': a.mean(), 'media_unico': b.mean(),
            'mediana_rec': a.median(), 'mediana_unico': b.median(),
            't': t, 'p_welch': p_welch, 'p_mannwhitney': p_mw,
            'd_cohen': (a.mean() - b.mean()) / desvio_comb}

resultados = {m: comparar(m) for m in ['gasto_total', 'ticket_medio']}

for metrica, r in resultados.items():
    print(f"=== {metrica} ===")
    print(f"Recorrentes: n={r['n_recorrentes']:,} | média={r['media_rec']:,.2f} | mediana={r['mediana_rec']:,.2f}")
    print(f"Compra única: n={r['n_unicos']:,} | média={r['media_unico']:,.2f} | mediana={r['mediana_unico']:,.2f}")
    print(f"Welch: t={r['t']:.2f}, p={r['p_welch']:.3g} | Mann-Whitney: p={r['p_mannwhitney']:.3g} | d de Cohen={r['d_cohen']:.2f}")
    print()

## 7. Resumo automático dos resultados

A célula abaixo monta o texto das conclusões **a partir dos números calculados** (nenhuma conclusão foi escrita à mão). Use esta saída para preencher o README.

In [ ]:
def conclusao(r):
    if r['p_welch'] < ALFA and r['p_mannwhitney'] < ALFA:
        return 'recorrentes gastam significativamente MAIS (os dois testes concordam)'
    if r['p_welch'] < ALFA or r['p_mannwhitney'] < ALFA:
        return 'evidência MISTA: um teste é significativo e o outro não (reportar com cautela)'
    return 'SEM evidência de que recorrentes gastem mais (não rejeita H0)'

def magnitude(d):
    d = abs(d)
    return 'desprezível' if d < 0.2 else 'pequeno' if d < 0.5 else 'médio' if d < 0.8 else 'grande'

print('=== NÚMEROS PARA O README ===')
for k, v in kpis.items():
    print(f'- {k}: {v}')
print(f'- Itens que são produtos: {len(produtos):,}; top 10 concentram {share_top10:.1%} da receita')
for sku, linha in top10.head(3).iterrows():
    print(f"- Top item: {linha['descricao']} = US$ {linha['receita']:,.2f}")
for nome, linha in seg.iterrows():
    print(f"- Segmento {nome}: {linha['% clientes']:.1f}% dos clientes, {linha['% receita']:.1f}% da receita")

pico = mensal.idxmax()
print(f"- Mês de maior receita: {pico:%m/%Y} (US$ {mensal.max():,.2f})")

print()
print('=== TESTE DE HIPÓTESE ===')
for metrica, r in resultados.items():
    aviso = ' [diferença estatística, mas de pouca relevância prática]' if abs(r['d_cohen']) < 0.2 else ''
    print(f"- {metrica}: {conclusao(r)}; efeito {magnitude(r['d_cohen'])} (d={r['d_cohen']:.2f}){aviso}; "
          f"mediana {r['mediana_rec']:,.2f} vs {r['mediana_unico']:,.2f}")

## 8. Limitações e próximos passos

- **Receita líquida aproximada:** o pareamento compra × cancelamento exige mesmo cliente, item e quantidade; devoluções parciais não são capturadas.
- **Cortes de frequência** (1, 2-3, 4-9, 10+) são arbitrários; vale testar alternativas.
- **Correlação, não causalidade:** o teste compara grupos observados, não prova que fidelização *causa* maior gasto.
- **Fora de escopo, mas recomendado:** segmentação **RFM** (Recência, Frequência, Valor), clusterização (K-Means), análise de cohort/retenção e um dashboard interativo.